# Example 1.2.3 / 4: Logical Parameters

Companion notebook for **1.2.3 Symbolic and Logical Parameters** in the AMPL lab Documentation, section "Logical Parameters". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Logical Parameters

AMPL has no true/false type, but a `binary` parameter (0 or 1) does the same job. Here some
products are promoted, and promoted products need a higher minimum inventory:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                          # products
param fr_reg > 0;                  # regular inventory fraction
param fr_pro > fr_reg;             # fraction for promoted products
param promote {PROD} binary;       # 1 if the product is promoted
param market {PROD} >= 0;          # most that can be sold, in tons
# minimum inventory: the promoted fraction for promoted products
param mininv {p in PROD} = (if promote[p] then fr_pro else fr_reg) * market[p];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["fr_reg"] = 0.1
ampl.param["fr_pro"] = 0.25
ampl.param["promote"] = {"bands": 1, "coils": 0, "plate": 0}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}

In [5]:
# run.py
print(ampl.param["mininv"].to_pandas())

       mininv
bands    1500
coils     400
plate     350


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param fr_reg := 0.1;
param fr_pro := 0.25;
param:  promote  market :=
  bands    1      6000
  coils    0      4000
  plate    0      3500 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display mininv;

mininv [*] :=
bands  1500
coils   400
plate   350
;

